In [1]:
import numpy as np
import pyvista as pv


def plot_data(
        data: np.ndarray | list,
        size: float = 5.0,
):
    coords = data
    colors = np.clip(data, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_clustered_pv(
        data: np.ndarray,
        labels: np.ndarray,
        cluster_centers: np.ndarray,
        size: float = 4.0
):
    coords = data
    cluster_colors = cluster_centers[labels]
    colors = np.clip(cluster_colors, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()


def plot_centroids_pv(
        cluster_centers: np.ndarray,
        size: float = 18.0
):
    coords = cluster_centers
    colors = np.clip(cluster_centers, 0, 1)

    cloud = pv.PolyData(coords)
    cloud["colors"] = (colors * 255).astype(np.uint8)

    plotter = pv.Plotter()
    plotter.add_points(
        cloud,
        scalars="colors",
        rgb=True,
        point_size=size,
        render_points_as_spheres=True
    )

    plotter.add_axes()
    plotter.show_grid()
    plotter.show()

In [2]:
import numpy as np

def generate_dataset(centers, stds, n_samples=500, ndim=3):
    X = []
    y = []

    for i, center in enumerate(centers):
        cluster = np.random.normal(loc=center, scale=stds[i], size=(n_samples // len(centers), ndim))
        X.append(cluster)
        y.append(np.full(n_samples // len(centers), i))

    return np.vstack(X), np.concatenate(y)

In [3]:
import pandas as pd
import pyvista as pv
import matplotlib.pyplot as plt

default_colors = np.array(plt.colormaps.get_cmap('tab10').colors)

In [4]:
data, ground_truth = generate_dataset(
    [[1, 1, 1], [3, 3, 3], [2, 2, 1]],
    stds=[.5, .5, .5],
    n_samples=500
)

pd.DataFrame(data)

,0,1,2
0,1.341243,0.607303,1.019970
1,2.160388,1.895812,0.832679
2,1.044507,0.468764,0.658997
3,2.285339,1.637357,1.211004
4,1.976452,0.953134,0.891941
...,...,...,...
493,2.049355,1.817761,-0.059034
494,2.439660,2.122769,0.786536
495,2.250190,2.164031,1.102738
496,1.490254,1.679159,1.547073


In [5]:
plt = pv.Plotter()

for n, d in zip(ground_truth, data):
    cloud = pv.PolyData(d)
    color = default_colors[n % len(default_colors)]

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=10
    )

plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:35233/index.html?ui=P_0x7fe40fad6270_0&reconnect=auto" class="pyvi…

In [6]:
from ktree.ntree import NTreeMean
from ktree.ntree import NTreeDynamic

tree = NTreeMean(2)

for a in data:
    tree.insert(a)

sorted_data = tree.sort()

In [7]:
plt = pv.Plotter()

for n, cluster in enumerate(sorted_data):
    s_data = list(cluster)
    cloud = pv.PolyData(s_data)
    color = default_colors[n % len(default_colors)]

    centroid = np.median(s_data, axis=0)

    plt.add_points(
        cloud,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=5
    )

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=len(s_data) // 2
    )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:35233/index.html?ui=P_0x7fe3ff1d6850_1&reconnect=auto" class="pyvi…

In [8]:
n_clusters = 3

all_clusters = sorted(sorted_data, key=lambda x: len(x))[::-1]
clusters = all_clusters[:n_clusters]
all_data = all_clusters[n_clusters:]

main_clusters = []

for cluster in clusters:
    c_centroid = np.mean([*cluster], axis=0)
    sum_dist = 0

    sub_cluster = []

    for data in all_data:
        d_centroid = np.median([*data], axis=0)
        dist = np.linalg.norm(c_centroid - d_centroid)

        sum_dist += dist

        sub_cluster.append((dist, data))

    mead_dist = sum_dist / len(all_data)
    sub_cluster = sorted(sub_cluster, key=lambda a: a[0])

    main_clusters.append((cluster, [data for (dist, data) in sub_cluster if dist < mead_dist]))


In [9]:
plt = pv.Plotter()

for n, (cluster, all_data) in enumerate(main_clusters):
    s_data = list(cluster)

    cloud = pv.PolyData(list(cluster))
    color = default_colors[n % len(default_colors)]

    centroid = np.mean(s_data, axis=0)

    plt.add_points(
        centroid,
        color=color,
        render_points_as_spheres=True,
        smooth_shading=True,
        point_size=25
    )

    for data in (all_data + [cluster]):
        cloud = pv.PolyData(list(data))

        plt.add_points(
            cloud,
            color=color,
            render_points_as_spheres=True,
            smooth_shading=True,
            point_size=10
        )


plt.add_axes()
plt.show_grid()
plt.show()

Widget(value='<iframe src="http://localhost:35233/index.html?ui=P_0x7fe3ff1d4cd0_2&reconnect=auto" class="pyvi…